# Ngày 21 · Dự án P3: Digit Recognizer

- Ghép PCA và KNN vào một Pipeline để PCA chỉ fit trên fold train
- Chọn số thành phần và k bằng stratified 5-fold có seed
- Xuất submission đúng định dạng ImageId, Label của Kaggle

**Đọc:**
- LeCun, Bottou, Bengio & Haffner (1998), Gradient-Based Learning Applied to Document Recognition, Proceedings of the IEEE 86(11), 2278–2324 (bộ MNIST)
- Alpaydin & Kaynak (1998), Optical Recognition of Handwritten Digits, UCI Machine Learning Repository (bộ load_digits)
- ISLR (bản 2), §12.2 (ôn) và §5.1.3

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))
from sklearn.datasets import load_digits
# Optical Recognition of Handwritten Digits (UCI), có sẵn trong scikit-learn: 1797 ảnh 8×8, giá trị 0–16.
X_dg, y_dg = load_digits(return_X_y=True)
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.exceptions import NotFittedError
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.utils.validation import check_is_fitted
_grid21 = [(5, 1), (10, 3), (20, 1), (30, 1)]
def _unfitted(pipe):
    try:
        check_is_fitted(pipe)
        return False
    except NotFittedError:
        return True
def _ref_cv21(make, nc, k, seed):
    return float(cross_val_score(make(nc, k), X_dg, y_dg, cv=StratifiedKFold(5, shuffle=True, random_state=seed)).mean())
def export_digits(make, n_components, k, limit=None):
    folder = DATA / 'kaggle/digit-recognizer'
    train, test = pd.read_csv(folder / 'train.csv', nrows=limit), pd.read_csv(folder / 'test.csv')
    pipe = make(n_components, k).fit(train.drop(columns='label').to_numpy(float), train.label.to_numpy())
    out = pd.DataFrame({'ImageId': np.arange(1, len(test) + 1), 'Label': pipe.predict(test.to_numpy(float))})
    out.to_csv('digits_submission.csv', index=False)
    return out


## Pipeline PCA + KNN và chọn cấu hình

Gợi ý: Pipeline([("pca", PCA(n_components, random_state=0)), ("knn", KNeighborsClassifier(n_neighbors=k))]). cv_digits dùng StratifiedKFold(5, shuffle=True, random_state=seed). select_config trả cặp có điểm cao nhất; hòa thì cặp đứng trước trong lưới.

In [ ]:
from sklearn.pipeline import Pipeline

def make_digits_pipeline(n_components, k):
    # Trả Pipeline CHƯA fit.
    raise NotImplementedError

def cv_digits(make, X, y, n_components, k, seed):
    raise NotImplementedError

def select_config(make, X, y, grid, seed):
    # grid: list các cặp (n_components, k). Trả cặp tốt nhất.
    raise NotImplementedError


In [ ]:
_report('d21-digits',
    _check('Pipeline chưa fit, có PCA rồi KNN đúng tham số', lambda: ((lambda p: _unfitted(p) and isinstance(p.steps[0][1], PCA) and p.steps[0][1].n_components == 12 and isinstance(p.steps[-1][1], KNeighborsClassifier) and p.steps[-1][1].n_neighbors == 4)(make_digits_pipeline(12, 4)), 'PCA phải nằm trong pipeline để fit lại trong từng fold.')),
    _check('cv_digits khớp stratified 5-fold tham chiếu', lambda: (np.isclose(cv_digits(make_digits_pipeline, X_dg, y_dg, 20, 3, 1), _ref_cv21(make_digits_pipeline, 20, 3, 1)), 'n_components = 20, k = 3, seed = 1.')),
    _check('select_config chọn đúng cặp tốt nhất', lambda: (select_config(make_digits_pipeline, X_dg, y_dg, _grid21, 0) == _grid21[int(np.argmax([_ref_cv21(make_digits_pipeline, nc, k, 0) for nc, k in _grid21]))], 'Lưới (5,1), (10,3), (20,1), (30,1).')),
)


## Dự án

Tải Digit Recognizer. Chạy select_config trên lưới n_components ∈ {20, 40, 80} × k ∈ {1, 3, 5} với 5000 ảnh train đầu (để chạy nhanh), rồi export_digits(make_digits_pipeline, nc, k) để ghi digits_submission.csv và tự nộp trên Kaggle. Báo cáo: bảng cấu hình → accuracy CV, điểm Kaggle, ma trận nhầm lẫn và 8 ảnh bị dự đoán sai. Tuần 4 sẽ so kết quả này với CNN bằng PyTorch.